# 01 — Calidad y EDA de `fact_award_item`

Este cuaderno es el espacio de exploración, gráficos, hipótesis y conclusiones de la Fase 2. Lee los resultados técnicos ya publicados; no modifica `data/processed/` ni reemplaza scripts reproducibles.

Antes de ejecutarlo deben existir:

- `data/processed/fact_award_item/_SUCCESS`
- `data/processed/fact_award/_SUCCESS`
- `artifacts/quality/fact_award_item/_SUCCESS`

Si falta el segundo, ejecute desde la raíz del repositorio:

    python -m src.quality.profile_fact_award_item --overwrite

## Regla financiera

`award_amount` no se carga ni se suma desde `fact_award_item`. Ese monto pertenece a una adjudicación y puede repetirse por cada ítem. La sección monetaria usa `fact_award`, que tiene una fila por adjudicación, y nunca mezcla monedas.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.dataset as ds
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

FACT_DIR = ROOT / 'data' / 'processed' / 'fact_award_item'
FACT_AWARD_DIR = ROOT / 'data' / 'processed' / 'fact_award'
PROFILE_DIR = ROOT / 'artifacts' / 'quality' / 'fact_award_item'

assert (FACT_DIR / '_SUCCESS').is_file(), f'Falta tabla publicada: {FACT_DIR}'
assert (FACT_AWARD_DIR / '_SUCCESS').is_file(), f'Falta tabla publicada: {FACT_AWARD_DIR}'
assert (PROFILE_DIR / '_SUCCESS').is_file(), f'Falta perfil publicado: {PROFILE_DIR}'

print(f'Raíz del proyecto: {ROOT}')


## 1. Perfil técnico publicado

Este resumen proviene del perfil reproducible. Sirve para decidir qué explorar; no sustituye la interpretación.


In [ ]:
with (PROFILE_DIR / 'profile.json').open(encoding='utf-8') as source:
    profile = json.load(source)

metrics = profile['global_metrics']
duplicates = profile['candidate_key_diagnostics']
profile_summary = pd.DataFrame(
    [
        ('Filas', metrics['rows']),
        ('Procesos distintos', metrics['distinct_processes']),
        ('Adjudicaciones: proceso + award_id', metrics['distinct_awards']),
        ('Proveedores distintos', metrics['distinct_suppliers']),
        ('Entidades distintas', metrics['distinct_entities']),
        ('Fechas de adjudicación ausentes', metrics['award_date_missing']),
        ('Clasificaciones de ítem ausentes', metrics['item_classification_id_missing']),
        ('Duplicados exactos excedentes', duplicates['exact_duplicate_excess_rows']),
    ],
    columns=['métrica', 'valor'],
)
display(profile_summary)

award_date_range = (
    f"**Rango de award_date:** {metrics['min_award_date']} a "
    f"{metrics['max_award_date']}"
)
display(Markdown(award_date_range))


## 2. Cargar una vista ligera para EDA

Se cargan solo columnas necesarias para calidad, tiempo y categorías. Para el volumen actual esta vista cabe en memoria; si el volumen aumenta, las agregaciones deben volver a Spark.


In [ ]:
analysis_columns = [
    'source_year',
    'process_ocid',
    'award_id',
    'supplier_id',
    'entity_id',
    'award_date',
    'award_currency',
    'procurement_category',
    'procurement_method',
    'item_classification_id',
]

fact_dataset = ds.dataset(str(FACT_DIR), format='parquet', partitioning='hive')
fact = fact_dataset.to_table(columns=analysis_columns).to_pandas()
fact['award_date'] = pd.to_datetime(fact['award_date'])

fact.head()


## 3. Cobertura por archivo, categoría, modalidad y moneda

Estas son frecuencias de filas al grano proceso × adjudicación × proveedor × ítem. Son adecuadas para cobertura y composición; no se interpretan como gasto.


In [ ]:
rows_by_source_year = (
    fact.groupby('source_year', dropna=False).size().rename('filas').reset_index()
)
category_counts = (
    fact['procurement_category'].fillna('<MISSING>').value_counts()
    .rename_axis('categoría')
    .reset_index(name='filas')
)
method_counts = (
    fact['procurement_method'].fillna('<MISSING>').value_counts()
    .rename_axis('modalidad')
    .reset_index(name='filas')
)
currency_counts = (
    fact['award_currency'].fillna('<MISSING>').value_counts()
    .rename_axis('moneda')
    .reset_index(name='filas')
)

display(rows_by_source_year)
display(category_counts)
display(method_counts)
display(currency_counts)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].bar(rows_by_source_year['source_year'].astype(str), rows_by_source_year['filas'])
axes[0, 0].set_title('Filas por source_year')
axes[0, 0].set_ylabel('Filas')

axes[0, 1].bar(category_counts['categoría'], category_counts['filas'])
axes[0, 1].set_title('Categoría de contratación')
axes[0, 1].tick_params(axis='x', rotation=25)

axes[1, 0].bar(method_counts['modalidad'], method_counts['filas'])
axes[1, 0].set_title('Modalidad de contratación')
axes[1, 0].tick_params(axis='x', rotation=25)

axes[1, 1].bar(currency_counts['moneda'], currency_counts['filas'])
axes[1, 1].set_title('Moneda de adjudicación')
axes[1, 1].tick_params(axis='x', rotation=25)

fig.tight_layout()


## 4. Montos por moneda al grano de adjudicación

`fact_award` tiene una fila por adjudicación. Por ello sus montos pueden agregarse, pero siempre se mantiene `award_currency` como dimensión: una suma entre monedas distintas no tiene interpretación económica.


In [ ]:
award_dataset = ds.dataset(str(FACT_AWARD_DIR), format='parquet', partitioning='hive')
award = award_dataset.to_table(
    columns=['source_year', 'award_amount', 'award_currency']
).to_pandas()
award['award_currency'] = award['award_currency'].fillna('<MISSING>')

amount_by_currency = (
    award.groupby('award_currency', dropna=False)
    .agg(adjudicaciones=('award_amount', 'size'), monto=('award_amount', 'sum'))
    .reset_index()
    .sort_values('adjudicaciones', ascending=False)
)
amount_by_source_year_currency = (
    award.groupby(['source_year', 'award_currency'], dropna=False)
    .agg(adjudicaciones=('award_amount', 'size'), monto=('award_amount', 'sum'))
    .reset_index()
    .sort_values(['source_year', 'award_currency'])
)

display(amount_by_currency)
display(amount_by_source_year_currency)


## 5. Tiempo y campos pendientes

`source_year` identifica el archivo descargado; el análisis temporal debe usar `award_date`. Las filas sin fecha se conservan, pero no podrán entrar a una división temporal hasta definir una regla explícita.


In [ ]:
missing_by_source_year = (
    fact.assign(
        award_date_missing=fact['award_date'].isna(),
        item_classification_missing=fact['item_classification_id'].isna(),
    )
    .groupby('source_year', dropna=False)[['award_date_missing', 'item_classification_missing']]
    .sum()
    .reset_index()
)
display(missing_by_source_year)

dated = fact.dropna(subset=['award_date']).assign(
    award_year=lambda frame: frame['award_date'].dt.year
)
coverage = pd.crosstab(dated['source_year'], dated['award_year'])
display(coverage)


## 6. Conclusiones iniciales y siguientes decisiones

Complete esta sección al interpretar los gráficos. Mantenga separadas las observaciones de las decisiones metodológicas.

- **Observación confirmada:** no hay duplicados exactos ni repeticiones de la llave candidata actual.
- **Observación confirmada:** existen varias monedas; no se sumarán montos sin conversión documentada.
- **Observación confirmada:** existen ítems sin clasificación; todavía no se eliminan.
- **Decisión confirmada:** `fact_award` es la tabla para métricas monetarias por moneda; `fact_award_item` se conserva para detalle por ítem.
- **Decisión confirmada:** `fact_award_supplier` conserva una fila por adjudicación–proveedor. En la corrida actual hay un proveedor por adjudicación; si cambia, no se asignarán montos por proveedor sin una regla explícita.
- **Decisión pendiente:** definir el tratamiento de las 35 fechas de adjudicación ausentes en modelos temporales.

Al cerrar el cuaderno, trasladar decisiones estables a `docs/README_Estrategico.md` y `docs/ContratoDatos.md`.
